# Stage 4 - Retrieve

## Data
Chroma DB

## Install
```

```

In [33]:
'''
1.1 Vectors DB Client
'''
import chromadb

client = chromadb.PersistentClient(path="chroma_kb") 
col = client.get_collection(name="knowledge_base")         

In [34]:
'''
1.2 Retrieve - dict
'''
from sentence_transformers import SentenceTransformer

THRESHOLD = 0.8
K = 5

embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="mps")

def retrieve_dict(query, col=col, embedder=embedder, k=K, threshold=THRESHOLD):
  """Retrieval-пайплайн: embed query -> search -> filter -> return."""
  q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
  res = col.query(query_embeddings=[q_vec.tolist()], n_results=k)
  out = []
  for doc, m, dist in zip(res["documents"][0],
                          res["metadatas"][0],
                          res["distances"][0]):
    sim = 1.0 - dist                        # Косинусное сходство
    if sim >= threshold:                    # Порог по сходству
      out.append({
        'id': m['id'],
        'doc_id': m['doc_id'], 
        'similarity': round(sim, 4), 
        'text': doc, 
      }
    )
  return out

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6798.56it/s]


In [ ]:
'''
1.3 load queries
'''
import json

with open("1_queries.json", encoding="utf-8") as f:
  data = json.load(f)

# filter GAPs
queries = [u for u in data if u.get("kind") != "gap"]

In [ ]:
'''
2. LLM Generate
'''
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8080/v1", api_key="not-needed")
LLM = "qwen3.5"                                       # Имя сервер игнорирует
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}


def generate(messages, max_tokens=160):
    """Один вызов локальной LLM. temperature=0 — для воспроизводимости,
    режим рассуждения Qwen3.5 отключён, поэтому сразу приходит готовый текст."""
    resp = client.chat.completions.create(
        model=LLM, messages=messages, temperature=0.0,
        max_tokens=max_tokens, extra_body=NO_THINK)
    return resp.choices[0].message.content.strip()

In [36]:
'''
6. Expansion and Ranking
'''
from collections import defaultdict

# def search(query, n=50):
#     """Поиск по тексту: кодирует его как query: и возвращает индексы
#     top-n документов по убыванию косинусного сходства."""
#     ret = retrieve(q["query"], col, embedder, k=5, threshold=0.0)
#     return ret


def expand(query, n=5):
  """Query expansion: n перефразировок запроса официальными словами базы."""
  text = generate([
      {"role": "system", "content": "Ты помощник поиска по базе знаний "
                                    "онлайн-кинотеатра."},
      {"role": "user", "content": f"Переформулируй запрос клиента {n} способами "
        f"официальными словами из базы знаний. Используй полные фразы (не одно "
        f"слово), синонимы и термины. По одной формулировке на строку, без "
        f"нумерации.\n\nЗапрос: {query}"}])
  expanded = [l.strip() for l in text.splitlines() if l.strip()][:n]
  # print('\n>>> expand: ', query, '\n', expanded)
  return expanded


def rrf(ranked_lists, k=60, top_k=5):
  """Reciprocal Rank Fusion: сливаем несколько выдач по рангам."""
  scores = defaultdict(float)
  for lst in ranked_lists:
      for rank, doc in enumerate(lst, start=1):
          scores[doc] += 1.0 / (k + rank)
  ranked = sorted(scores.items(), key=lambda x: -x[1])
  return [doc for doc, _ in ranked[:top_k]]

In [ ]:
'''
RRF Test on 3 queries
'''

# for qi in [30]:
for qi in [22, 30, 33]:
  query = queries[qi]
  rel = set(queries[qi]["relevant_docs"])

  # basic search
  basic = retrieve_dict(query['query'])
  # print('>>> basic: ', basic)

  # expanded search
  paras = expand(query['query'])
  # print('>>> paras: ', paras)
  exp = [retrieve_dict(p) for p in paras]
  exp.append(basic)

  tot = []
  for q in exp:
    r = [(i['id'], i['text'], i['similarity'], i['doc_id']) for i in q]
    tot.append(r)
  ranked = rrf(tot)
  print('>>> ranked: ', ranked)

  mark = lambda i: "+" if i in rel else " "
  print(f"\n[{qi}] {query['query']} {query['relevant_docs']}")
  
  for p in paras:
    print("   ·", p)

  print("  обычный поиск top5:",
    [f"{mark(i['doc_id'])}#{i['doc_id']}/{i['id']} {i['text'][:32]}" for i in basic])
  print("  query expansion top5:",
    [f"{mark(i[0])}#{i[0]} {i[1][:32]}" for i in ranked])

>>> ranked:  [(117, 'Такие обращения направляйте на help@filmoskop.ru с пометкой о том, что вопрос от организации, — письмо передадут нужной команде.', 0.8421, 103), (130, 'Позиция просмотра сохраняется автоматически и синхронизируется между устройствами аккаунта. Начали фильм на телефоне — на телевизоре он появится в разделе «Продолжить просмотр» на главном экране, и воспроизведение начнётся с того же места. Синхронизация занимает до минуты после паузы; чтобы позиция точно сохранилась, поставьте видео на паузу перед закрытием приложения. Позиция хранится отдельно для каждого профиля.', 0.8258, 111), (130, 'Позиция просмотра сохраняется автоматически и синхронизируется между устройствами аккаунта. Начали фильм на телефоне — на телевизоре он появится в разделе «Продолжить просмотр» на главном экране, и воспроизведение начнётся с того же места. Синхронизация занимает до минуты после паузы; чтобы позиция точно сохранилась, поставьте видео на паузу перед закрытием приложения. Позиция храни

In [53]:
'''
RRF Test on 3 queries - 2
'''

# for qi in [30]:
for qi in [22, 30, 33]:
  query = queries[qi]
  rel = set(queries[qi]["relevant_docs"])

  # basic search
  basic = retrieve_dict(query['query'], k=10)
  # print('>>> basic: ', basic)

  # expanded search
  paras = expand(query['query'])
  # print('>>> paras: ', paras)
  exp = [retrieve_dict(p) for p in paras]
  exp.append(basic)

  tot = []
  for q in exp:
    r = [(i['id'], i['text'], i['similarity'], i['doc_id']) for i in q]
    tot.append(r)
  ranked = rrf(tot, k=10)
  print('>>> ranked: ', ranked)

  mark = lambda i: "+" if i in rel else " "
  print(f"\n[{qi}] {query['query']} {query['relevant_docs']}")
  
  for p in paras:
    print("   ·", p)

  print("  обычный поиск top5:",
    [f"{mark(i['doc_id'])}#{i['doc_id']}/{i['id']} {i['text'][:32]}" for i in basic])
  print("  query expansion top5:",
    [f"{mark(i[0])}#{i[0]}/{i[3]} {i[1][:32]}" for i in ranked])

>>> ranked:  [(85, 'Раздел в разработке Мы готовим этот раздел. Информация появится в ближайшее время.', 0.8238, 74), (117, 'Такие обращения направляйте на help@filmoskop.ru с пометкой о том, что вопрос от организации, — письмо передадут нужной команде.', 0.8421, 103), (130, 'Позиция просмотра сохраняется автоматически и синхронизируется между устройствами аккаунта. Начали фильм на телефоне — на телевизоре он появится в разделе «Продолжить просмотр» на главном экране, и воспроизведение начнётся с того же места. Синхронизация занимает до минуты после паузы; чтобы позиция точно сохранилась, поставьте видео на паузу перед закрытием приложения. Позиция хранится отдельно для каждого профиля.', 0.8258, 111), (130, 'Позиция просмотра сохраняется автоматически и синхронизируется между устройствами аккаунта. Начали фильм на телефоне — на телевизоре он появится в разделе «Продолжить просмотр» на главном экране, и воспроизведение начнётся с того же места. Синхронизация занимает до минуты после па

In [22]:
def basic_search(query):
  ret = retrieve_dict(query)
  doc_ids = [i['doc_id'] for i in ret]
  return doc_ids

In [49]:
def precision_at_k(found_ids, relevant_ids, k):
    """Доля релевантных среди k выданных чанков."""
    hits = [i for i in found_ids[:k] if i in relevant_ids]
    return len(hits) / k


def reciprocal_rank(found_ids, relevant_ids):
    """Обратный ранг первого релевантного чанка; 0, если его нет в выдаче."""
    for rank, i in enumerate(found_ids, start=1):
        if i in relevant_ids:
            return 1.0 / rank
    return 0.0

In [ ]:
def retrieve_hyde(query, answer, col, embedder, k=5, threshold=0.0):
  """Retrieval-пайплайн: embed query -> search -> filter -> return."""
  q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
  q_ans = embedder.encode("query: " + answer, normalize_embeddings=True)
  vec = q_vec + q_ans
  vec = vec / np.linalg.norm(vec)

  res = col.query(query_embeddings=[vec.tolist()], n_results=k)
  out = []
  for doc, m, dist in zip(res["documents"][0],
                          res["metadatas"][0],
                          res["distances"][0]):
    sim = 1.0 - dist                        # Косинусное сходство
    if sim >= threshold:                    # Порог по сходству
      out.append((m['doc_id'], doc, sim, m['id']))
  
  return out


def retrieve_hyde_dict(query, answer, k=5, threshold=0.0):
  """Retrieval-пайплайн: embed query -> search -> filter -> return."""
  q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
  q_ans = embedder.encode("query: " + answer, normalize_embeddings=True)
  vec = q_vec + q_ans
  vec = vec / np.linalg.norm(vec)

  res = col.query(query_embeddings=[vec.tolist()], n_results=k)
  out = []
  for doc, m, dist in zip(res["documents"][0],
                          res["metadatas"][0],
                          res["distances"][0]):
    sim = 1.0 - dist                        # Косинусное сходство
    if sim >= threshold:                    # Порог по сходству
      out.append({
        'id': m['id'],
        'doc_id': m['doc_id'], 
        'similarity': round(sim, 4), 
        'text': doc, 
      }
    ) 
  return out


def hyde(query, top_k=5):
    """HyDE: ищем по среднему векторов запроса и гипотетического ответа."""
    doc = generate([
        {"role": "system", "content": "Ты пишешь короткие ответы из базы знаний "
                                      "онлайн-кинотеатра."},
        {"role": "user", "content": f"Напиши короткий (1-2 предложения) фрагмент-"
         f"ответ из базы знаний на вопрос клиента официальным языком. Только "
         f"текст ответа.\n\nВопрос: {query}"}], max_tokens=110)
    hits = retrieve_hyde(query, doc, col, embedder, top_k)
    return doc, hits


for qi in [22, 30, 33]:
    query = queries[qi]["query"]
    rel = set(queries[qi]["relevant_docs"])
    doc, hits = hyde(query)
    mark = lambda i: "+" if i in rel else " "

    found = retrieve_dict(query)
    # print('>>> found: ', found)

    print(f"\n[{qi}] {query}")
    print("  гипотетический ответ:", doc)
    print("  обычный поиск top5:",
        [f"{mark(i['doc_id'])}#{i['doc_id']} {i['text'][:32]}" for i in found])
    print("  HyDE top5:", [f"{mark(i[0])}#{i[3]} {i[1][:32]}" for i in hits]) 

NameError: name 'retrieve' is not defined

In [66]:
'''
4. Retrieve func
'''
from sentence_transformers import SentenceTransformer

THRESHOLD = 0.8

embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="mps")

def retrieve(query, col=col, embedder=embedder, k=5, threshold=THRESHOLD):
  """Retrieval-пайплайн: embed query -> search -> filter -> return."""
  q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
  res = col.query(query_embeddings=[q_vec.tolist()], n_results=k)
  out = []
  for doc, m, dist in zip(res["documents"][0],
                          res["metadatas"][0],
                          res["distances"][0]):
    sim = 1.0 - dist                        # Косинусное сходство
    if sim >= threshold:                    # Порог по сходству
      out.append((m['doc_id'], doc, sim, m['id']))
  
  return out

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6938.15it/s]


In [71]:
import numpy as np

K = 10

# def recall_at_k(found, relevant, k=5):
#   """Доля релевантных документов в первых k результатах."""
#   hits = set(found[:k]) & relevant
#   return len(hits) / len(relevant) if relevant else 0.0

rows = {"Обычный поиск": [], "Query expansion": [], "HyDE": []}

for q in queries[:]:
    query, rel = q["query"], set(q["relevant_docs"])

    # Basic
    base = retrieve_dict(query, k=K)
    base_docs = [t['doc_id'] for t in base]
    rows["Обычный поиск"].append(recall_at_k(base_docs, rel))

    # Exxpanded
    # exp = [base] + [retrieve_dict(p) for p in expand(query)]
    # tot = []
    # for q in exp:
    #   r = [(i['id'], i['text'], i['similarity'], i['doc_id']) for i in q]
    #   tot.append(r)
    # ranked = rrf(r)
    # # print('>>> ranked: ', ranked)
    # all_flat = [item for sublist in exp for item in sublist]
    # # print('>>> all_flat: ', all_flat)
    # ranked_items = [i for i in all_flat if i.get("id") in ranked]
    # exp_docs = [t['doc_id'] for t in ranked_items]
    # rows["Query expansion"].append(recall_at_k(exp_docs, rel))

    exp = rrf([retrieve(query, k=K)] + [retrieve(p, k=K) for p in expand(query)])
    exp_docs = [t[0] for t in exp]
    rows["Query expansion"].append(recall_at_k(exp_docs, rel))

    # HyDE
    _, hyd = hyde(query, K)
    hyd_docs = [t[0] for t in hyd]
    rows["HyDE"].append(recall_at_k(hyd_docs, rel))

n = len(queries)
print(f"{'Подход':<18}{'recall@5':>10}")
for name in rows:
  print(f"{name:<18}{np.mean(rows[name]):>10.3f}")

Подход              recall@5
Обычный поиск          0.723
Query expansion        0.634
HyDE                   0.804


# Sparse search

## install
pip install rank-bm25

In [ ]:
import re
from collections import defaultdict
from rank_bm25 import BM25Okapi

def tokenize(text):
    """Разбивает текст на токены-слова в нижнем регистре — вход для BM25."""
    return re.findall(r"\w+", text.lower())


bm25 = BM25Okapi([tokenize(t) for t in texts])


def sparse_search(query, k=3):
    """Разреженный поиск BM25: top-k по совпадению слов с весом по редкости."""
    scores = bm25.get_scores(tokenize(query))
    order = np.argsort(scores)[::-1][:k]
    return [int(i) for i in order]


def rrf_fuse(ranked_lists, rrf_k=60):
    """Reciprocal Rank Fusion: сливаем выдачи по рангам."""
    scores = defaultdict(float)
    for lst in ranked_lists:
        for rank, doc in enumerate(lst, start=1):
            scores[doc] += 1.0 / (rrf_k + rank)
    return [doc for doc, _ in sorted(scores.items(), key=lambda x: -x[1])]


def hybrid_search(query, k=3):
    """Гибридный поиск: сливаем плотную и разреженную выдачи через RRF."""
    fused = rrf_fuse([dense_search(query, 15), sparse_search(query, 15)])
    return fused[:k]


def is_hit(chunk_ids, relevant_ids):
    """Успех, если релевантный чанк дошёл до контекста (retrieval-hit) —
    метрика из задания 4."""
    return any(i in relevant_ids for i in chunk_ids)


methods = {"dense": retrieve_dict, "BM25": sparse_search}
# methods = {"dense": retrieve_dict, "BM25": sparse_search, "hybrid": hybrid_search}
score = {m: 0 for m in methods}
print(f"{'тип':10}{'запрос':46}" + "".join(f"{m:>8}" for m in methods))
for q in queries[:3]:
    row = f"{q['kind']:10}{q['query'][:44]:46}"
    for m, fn in methods.items():
        answer, ids, *_ = rag_answer(q["query"], fn, k=3)
        ok = is_hit(ids, q["relevant_docs"])
        score[m] += ok
        row += f"{('верно' if ok else '—'):>8}"
    print(row)

n = len(queries)
print("\nВерных ответов из", n, ":", {m: score[m] for m in methods})